# Intention Collapse: Experiments v4.0

## Robust Checkpointing Architecture

This notebook uses:
- `shared_utils.py` - Experiment logic (metrics, probes, evaluation)
- `checkpoint_utils.py` - Robust checkpointing (JSONL, resume, error handling)

### Features
- **Auto-resume**: If Colab disconnects, just re-run - it continues from where it stopped
- **Per-item saving**: Each result is saved immediately (no data loss)
- **Error tolerance**: Single item errors don't abort the run

### Execution
1. Configure `MODEL_FAMILY` and `BENCHMARK` in Section 2
2. Run all cells
3. If interrupted, just run all cells again - it will resume
4. Repeat for all 9 combinations

---
## 1. Setup

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate bitsandbytes
!pip install -q datasets scikit-learn scipy sympy
!pip install -q matplotlib seaborn tqdm

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PATH = '/content/drive/MyDrive/intention_collapse_v4'
SPLITS_PATH = os.path.join(DRIVE_PATH, 'splits')
CHECKPOINTS_PATH = os.path.join(DRIVE_PATH, 'checkpoints')
os.makedirs(DRIVE_PATH, exist_ok=True)
os.makedirs(SPLITS_PATH, exist_ok=True)
os.makedirs(CHECKPOINTS_PATH, exist_ok=True)
print(f"✓ Drive path: {DRIVE_PATH}")
print(f"✓ Checkpoints: {CHECKPOINTS_PATH}")

In [ ]:
# Clone repository from GitHub
import os
from pathlib import Path
import shutil

# Remove repo if exists
if Path('intention-collapse-experiments').exists():
    shutil.rmtree('intention-collapse-experiments')

# Clone from GitHub
print("Cloning repository from GitHub...")
!git clone -q https://github.com/patriciomvera/intention-collapse-experiments.git

# Navigate to repository root
os.chdir('intention-collapse-experiments')

# Verify files exist
assert Path('src/shared_utils.py').exists(), "❌ shared_utils.py not found!"
assert Path('src/checkpoint_utils.py').exists(), "❌ checkpoint_utils.py not found!"

print(f"✓ Working directory: {os.getcwd()}")
print(f"✓ Repository cloned successfully")

# Add src to Python path
import sys
sys.path.insert(0, 'src')

In [ ]:
# Import modules
import shared_utils as U
import checkpoint_utils as C

# Get versions robustly (in case __version__ is missing)
CHECKPOINT_VERSION = getattr(C, '__version__', getattr(C, 'CODE_VERSION', 'unknown'))

print(f"✓ shared_utils v{U.CODE_VERSION}")
print(f"✓ checkpoint_utils v{CHECKPOINT_VERSION}")
print(f"\nEnvironment: {U.get_environment_info()}")

In [ ]:
# Run sanity checks
print("Running sanity checks...")
checks_ok = U.run_sanity_checks()

if not checks_ok:
    raise RuntimeError("Sanity checks failed! Fix issues before running experiments.")

# Checkpoint sanity test
print("\nRunning checkpoint sanity test...")
C.run_sanity_test()

In [ ]:
# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("GPU required! Enable in Runtime > Change runtime type")

In [ ]:
# Setup HuggingFace token
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("✓ HF_TOKEN loaded from Colab Secrets")
except:
    import getpass
    HF_TOKEN = getpass.getpass("Enter HuggingFace token: ")

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)

---
## 2. ⚙️ CONFIGURATION

In [ ]:
# =============================================================================
# ⚙️ CONFIGURATION - CHANGE THESE VALUES FOR EACH RUN
# =============================================================================

MODEL_FAMILY = 'mistral'  # 'mistral', 'llama', or 'qwen'
BENCHMARK = 'gsm8k'       # 'gsm8k', 'math', or 'arc'

# =============================================================================
# Other settings (usually keep default)
# =============================================================================

CONFIG = {
    'subset_size': 200,
    'seed': 42,
    'quantization': '4bit',

    # Generation
    'max_new_tokens_baseline': 200,  # v3.1: increased from 50 to prevent truncation
    'max_new_tokens_cot': 512,
    'max_new_tokens_babble': 200,

    # Metrics
    'variance_threshold': 0.90,
    'entropy_top_k': 100,

    # Paths
    'drive_path': DRIVE_PATH,
    'splits_path': SPLITS_PATH,
    'checkpoints_path': CHECKPOINTS_PATH,
    
    # Checkpointing
    'sync_interval': 10,  # Save progress every N items
    'fsync_every': 25,    # Force disk sync every N writes (higher = faster but less safe)
}

# Unique run ID for this experiment (DO NOT CHANGE between resume attempts!)
RUN_ID = f"{MODEL_FAMILY}_{BENCHMARK}_seed{CONFIG['seed']}"

# Set all seeds
U.set_all_seeds(CONFIG['seed'])

print("="*60)
print(f"📊 EXPERIMENT: {MODEL_FAMILY.upper()} on {BENCHMARK.upper()}")
print("="*60)
print(f"  Model: {U.MODEL_CONFIGS[MODEL_FAMILY]['name']}")
print(f"  Problems: {CONFIG['subset_size']}")
print(f"  Run ID: {RUN_ID}")
print(f"  Version: {U.CODE_VERSION}")
print("="*60)

# Check existing progress
C.print_run_status(CHECKPOINTS_PATH, RUN_ID)

---
## 3. Load Model and Data

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_config = U.MODEL_CONFIGS[MODEL_FAMILY]
print(f"Loading model: {model_config['name']}")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_config['name'], token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_config['name'],
    quantization_config=quantization_config,
    device_map="auto",
    token=HF_TOKEN,
    torch_dtype=torch.float16
)
model.eval()

print(f"\n✓ Model loaded!")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Layers: {model.config.num_hidden_layers}")

In [ ]:
# Load dataset with consistent indices (same problems for all models)
print(f"\nLoading {BENCHMARK} dataset...")

# Get or create indices (ensures reproducibility across models)
# IMPORTANT: Do NOT change these between resume attempts!
indices = U.get_or_create_indices(
    benchmark=BENCHMARK,
    subset_size=CONFIG['subset_size'],
    seed=CONFIG['seed'],
    splits_dir=CONFIG['splits_path']
)

# Load problems
problems = U.load_problems(BENCHMARK, indices)

print(f"✓ Loaded {len(problems)} problems")
print(f"  Original indices: {indices[:5]}... (saved for reproducibility)")
print(f"\nExample: {problems[0].question[:150]}...")

---
## 4. Run Experiment (with Checkpointing)

This section will:
- Automatically resume if previously interrupted
- Save each result immediately to JSONL
- Handle errors without aborting

In [ ]:
from tqdm.auto import tqdm
import numpy as np

# Create activation extractor
extractor = U.ActivationExtractor(model, model_config['extraction_layers'])

# Token limits per condition
max_tokens_map = {
    'baseline': CONFIG['max_new_tokens_baseline'],
    'enhanced': CONFIG['max_new_tokens_cot'],
    'babble': CONFIG['max_new_tokens_babble']
}

conditions = ['baseline', 'enhanced', 'babble']
summaries = {}

print("="*60)
print(f"RUNNING: {MODEL_FAMILY.upper()} on {BENCHMARK.upper()}")
print(f"Run ID: {RUN_ID}")
print("="*60)

for condition in conditions:
    max_tokens = max_tokens_map[condition]
    
    # Create runner for this condition
    runner = C.ExperimentRunner(
        run_id=RUN_ID,
        output_dir=CHECKPOINTS_PATH,
        condition=condition,
        sync_interval=CONFIG['sync_interval']
    )
    
    # Start runner and get resume point
    start_from = runner.start(len(problems))
    
    # Iterate with manual index control (NOT using list() on generator!)
    for i in tqdm(
        range(start_from, len(problems)),
        desc=condition,
        total=len(problems),
        initial=start_from
    ):
        problem = problems[i]
        runner.set_current_idx(i)  # Keep checkpoint idx aligned
        
        try:
            # Run the model
            result, acts = U.run_single_problem(
                model=model,
                tokenizer=tokenizer,
                extractor=extractor,
                problem=problem,
                condition=condition,
                max_new_tokens=max_tokens,
                entropy_top_k=CONFIG['entropy_top_k']
            )
            
            # Save immediately (result.to_dict() is JSON-serializable)
            runner.save_item(result, acts)
            
        except Exception as e:
            # Record error but continue
            runner.record_error(i, e)
    
    # Finalize this condition
    summaries[condition] = runner.finalize()

print("\n" + "="*60)
print("✅ ALL CONDITIONS COMPLETE")
print("="*60)

---
## 5. Load Results from Checkpoints

Load the saved JSONL files for analysis

In [ ]:
import json
from pathlib import Path

# Load results from JSONL files
results = {}
activations = {}

for condition in conditions:
    # Load results
    results_path = Path(CHECKPOINTS_PATH) / f"{RUN_ID}_{condition}_results.jsonl"
    acts_path = Path(CHECKPOINTS_PATH) / f"{RUN_ID}_{condition}_activations.npz"
    
    # Check if results file exists
    if not results_path.exists():
        print(f"⚠️ Missing results file: {results_path.name}")
        results[condition] = []
        activations[condition] = {'arrays': [], 'idxs': []}
        continue
    
    # Read JSONL
    condition_results = []
    with open(results_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                # Skip error placeholders for analysis
                if obj.get('status') != 'error':
                    condition_results.append(obj)
            except json.JSONDecodeError:
                continue  # Skip malformed lines
    
    results[condition] = condition_results
    print(f"{condition}: {len(condition_results)} successful results")
    
    # Load activations with indices
    if acts_path.exists():
        data = np.load(str(acts_path))
        acts_array = data['activations']
        acts_idxs = data['idxs'].tolist() if 'idxs' in data else list(range(len(acts_array)))
        activations[condition] = {
            'arrays': [acts_array[i] for i in range(len(acts_array))],
            'idxs': acts_idxs
        }
        print(f"  Activations: {len(acts_idxs)} arrays")
    else:
        activations[condition] = {'arrays': [], 'idxs': []}

---
## 6. Compute Metrics

In [ ]:
# Aggregate results
print("\nAggregating results...")

aggregated = {}

for condition in conditions:
    res_list = results[condition]
    acts_list = activations[condition]['arrays']
    
    # Extract values from dicts
    n = len(res_list)
    if n == 0:
        continue
    
    # Basic metrics
    if condition != 'babble':
        accuracies = [r['is_correct'] for r in res_list]
        accuracy = np.mean(accuracies)
        accuracy_std = np.std(accuracies)
    else:
        accuracy = None
        accuracy_std = None
    
    entropies = [r['metrics']['entropy'] for r in res_list]
    gen_tokens = [r['generated_tokens'] for r in res_list]
    
    # Dimensionality (if activations available)
    dim_eff_global = 0
    dim_eff_layerwise = []
    
    if acts_list:
        acts_array = np.stack(acts_list, axis=1)  # (n_layers, n_samples, hidden_dim)
        dim_eff_layerwise = U.compute_layer_wise_dim_eff(acts_array, CONFIG['variance_threshold'])
        dim_eff_global = int(np.median(dim_eff_layerwise))
    
    aggregated[condition] = {
        'n_samples': n,
        'accuracy': accuracy,
        'accuracy_std': accuracy_std,
        'entropy_mean': float(np.mean(entropies)),
        'entropy_std': float(np.std(entropies)),
        'dim_eff_global': dim_eff_global,
        'dim_eff_layerwise': dim_eff_layerwise,
        'generated_tokens_mean': float(np.mean(gen_tokens)),
        'generated_tokens_std': float(np.std(gen_tokens)),
    }
    
    print(f"\n{condition.upper()}:")
    if accuracy is not None:
        print(f"  Accuracy: {accuracy:.1%}")
    print(f"  Entropy: {aggregated[condition]['entropy_mean']:.3f} ± {aggregated[condition]['entropy_std']:.3f}")
    print(f"  dim_eff (global): {dim_eff_global}")
    print(f"  dim_eff (layers): {dim_eff_layerwise}")
    print(f"  Tokens generated: {aggregated[condition]['generated_tokens_mean']:.1f}")

In [ ]:
# Train recoverability probes
print("\nTraining recoverability probes...")

probe_results = {}

for condition in ['baseline', 'enhanced']:
    acts_data = activations[condition]
    res_list = results[condition]
    
    if not acts_data['arrays'] or not res_list:
        continue
    
    # Build aligned arrays (matching activations to results by idx)
    aligned_acts = []
    aligned_labels = []
    
    for i, idx in enumerate(acts_data['idxs']):
        # Find matching result
        for r in res_list:
            if r.get('problem_idx') == idx or r.get('_checkpoint_idx') == idx:
                aligned_acts.append(acts_data['arrays'][i])
                aligned_labels.append(r['is_correct'])
                break
    
    if not aligned_acts:
        print(f"  {condition}: No aligned data for probe")
        continue
    
    acts = np.stack(aligned_acts, axis=0)
    labels = np.array(aligned_labels)
    
    probe = U.train_recoverability_probe(
        acts, labels,
        cv_folds=5,
        n_bootstrap=1000
    )
    probe_results[condition] = probe
    
    print(f"\n{condition.upper()}:")
    print(f"  AUROC: {probe.auroc:.3f} [{probe.auroc_ci_low:.3f}, {probe.auroc_ci_high:.3f}]")
    print(f"  Balanced Acc: {probe.balanced_acc:.3f}")
    print(f"  Brier: {probe.brier:.3f}")
    print(f"  Samples: {probe.n_samples}, Pos rate: {probe.pos_rate:.1%}")

---
## 7. Save Final Results

In [ ]:
from datetime import datetime
from dataclasses import asdict

# Prepare final summary
final_summary = {
    'version': U.CODE_VERSION,
    'checkpoint_version': CHECKPOINT_VERSION,
    'environment': U.get_environment_info(),
    'run_id': RUN_ID,
    'model_family': MODEL_FAMILY,
    'model_name': U.MODEL_CONFIGS[MODEL_FAMILY]['name'],
    'benchmark': BENCHMARK,
    'n_problems': len(problems),
    'dataset_indices': indices,
    'timestamp': datetime.now().isoformat(),
    'config': CONFIG,
    'conditions': {},
    'summaries': summaries,  # From checkpoint runner
}

# Add aggregated results per condition
for condition in conditions:
    final_summary['conditions'][condition] = aggregated.get(condition, {})
    
    if condition in probe_results:
        final_summary['conditions'][condition]['probe'] = asdict(probe_results[condition])

# Save final JSON
json_path = os.path.join(DRIVE_PATH, f"{MODEL_FAMILY}_{BENCHMARK}_results.json")
with open(json_path, 'w') as f:
    json.dump(final_summary, f, indent=2, default=str)

print(f"\n✓ Final results saved: {json_path}")

# Copy activations to main drive path (consolidated)
for condition in conditions:
    src_path = Path(CHECKPOINTS_PATH) / f"{RUN_ID}_{condition}_activations.npz"
    if src_path.exists():
        dst_path = Path(DRIVE_PATH) / f"{MODEL_FAMILY}_{BENCHMARK}_{condition}_activations.npz"
        import shutil
        shutil.copy(str(src_path), str(dst_path))
        print(f"  ✓ Activations: {dst_path.name}")

---
## 8. Quick Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Accuracy
ax = axes[0]
accs = [aggregated.get(c, {}).get('accuracy', 0) or 0 for c in ['baseline', 'enhanced']]
bars = ax.bar(['Baseline', 'CoT'], accs, color=['steelblue', 'coral'])
ax.set_ylabel('Accuracy')
ax.set_title(f'{MODEL_FAMILY.upper()} on {BENCHMARK.upper()}: Accuracy')
ax.set_ylim(0, 1)
for i, v in enumerate(accs):
    ax.text(i, v + 0.02, f'{v:.1%}', ha='center')

# Entropy
ax = axes[1]
ents = [aggregated.get(c, {}).get('entropy_mean', 0) for c in conditions]
ax.bar(['Baseline', 'CoT', 'Babble'], ents, color=['steelblue', 'coral', 'gray'])
ax.set_ylabel('Entropy (bits)')
ax.set_title('Pre-collapse Intention Entropy')

# Probe AUROC
ax = axes[2]
aurocs = [probe_results[c].auroc if c in probe_results else 0.5 for c in ['baseline', 'enhanced']]
ci_low = [probe_results[c].auroc_ci_low if c in probe_results else 0.5 for c in ['baseline', 'enhanced']]
ci_high = [probe_results[c].auroc_ci_high if c in probe_results else 0.5 for c in ['baseline', 'enhanced']]
yerr = [[a - l for a, l in zip(aurocs, ci_low)], [h - a for a, h in zip(aurocs, ci_high)]]
ax.bar(['Baseline', 'CoT'], aurocs, yerr=yerr, capsize=5, color=['steelblue', 'coral'])
ax.axhline(0.5, color='gray', linestyle='--', label='Chance')
ax.set_ylabel('AUROC')
ax.set_title('Recoverability Probe')
ax.set_ylim(0, 1)

plt.tight_layout()

# Save figure
fig_path = os.path.join(DRIVE_PATH, f"{MODEL_FAMILY}_{BENCHMARK}_summary.png")
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"\n✓ Figure saved: {fig_path}")
plt.show()

---
## 9. Experiment Status

In [ ]:
# Show all completed experiments
available = U.list_available_results(DRIVE_PATH)

print("\nCompleted experiments:")
print("-" * 40)

# Create completion matrix
models = ['mistral', 'llama', 'qwen']
benchmarks = ['gsm8k', 'math', 'arc']

print(f"{'Model':<12} | {'GSM8K':<8} | {'MATH':<8} | {'ARC':<8}")
print("-" * 45)

for m in models:
    row = [m.upper()]
    for b in benchmarks:
        if (m, b) in available:
            row.append('✓')
        else:
            row.append('○')
    print(f"{row[0]:<12} | {row[1]:<8} | {row[2]:<8} | {row[3]:<8}")

print(f"\nTotal: {len(available)}/9 completed")

---
## Done! 🎉

### Next steps:
1. Change `MODEL_FAMILY` and `BENCHMARK` in Section 2
2. **Just run all cells again** - it will resume if needed
3. Repeat for all 9 combinations

### If Colab disconnects:
- Don't panic! Your progress is saved.
- Reconnect and run all cells - it will resume automatically.

After all 9 experiments: run `02_consolidate_results.ipynb`